<a href="https://colab.research.google.com/github/fayrouzhassan2000/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fayrouzhassan2000/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Getting The Data

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

# Get the token securely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

# Hugging Face dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Check the number of rows in the daily performance table
con.sql(f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one pseudonymized content item for one pseudonymized client on one report date. I will use March 2026 as the development time window, with the decision moment at the end of March 2026. The main table is `fact_content_daily_performance`, with `dim_content` joined when content attributes such as `word_count` are needed. The prediction target is a proxy for content decline/review priority, defined using the following month's performance. I deliberately exclude future outcome information, such as April performance, from the model features to prevent data leakage.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

**Features**
- `gsc_impressions` — historical search performance.
- `gsc_clicks` — historical search performance.
- `gsc_avg_position` — historical search ranking performance.
- `ga4_sessions` — historical analytics performance when GA4 data is available.
- `content_type` — a content attribute from `dim_content`.

**Label / Proxy**
- `is_declining_label` — a proxy target for identifying content that may need review.

**Context**
- `report_date` — identifies the observation date.
- `client_hash_id` — identifies the client and supports joins.
- `content_hash_id` — identifies the content item and supports joins.
- `gsc_data_available` — indicates whether GSC data is available.
- `ga4_data_available` — indicates whether GA4 data is available.

**Excluded**
- Fields that are not available at the decision moment or that would introduce target leakage will be excluded from the feature set.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain

The expected grain is one row per `report_date`, `client_hash_id`, and `content_hash_id`.

I check whether any combination appears more than once in the March 2026 slice.

In [23]:
q1 = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
""")

q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

No duplicate combinations were found, supporting the expected grain of one row per report date, client, and content item.

### Query 2 — Row count and date span

This query checks the number of rows and the date range of the March 2026 slice.

In [24]:
q2 = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

q2

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

The March 2026 slice contains 984,138 rows and spans from 2026-03-01    to        2026-03-31.

### Query 3 — GSC availability

This query checks how many rows in the March 2026 slice have GSC data available, using `IS TRUE`.

In [25]:
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS gsc_available_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""")

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ gsc_available_rows │
│       int64        │
├────────────────────┤
│            3611061 │
└────────────────────┘

### Five-feature frame

The decision moment is the end of March 2026. The features use information available by that point and are built from the March 2026 slice.

In [4]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY
        client_hash_id,
        content_hash_id
""")

features

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬────────────────────┬──────────────┐
│     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ ga4_sessions │
│         varchar         │         varchar          │     int128      │   int128   │       double       │    int128    │
├─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼────────────────────┼──────────────┤
│ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │            6523 │          7 │  7.209549402968802 │            1 │
│ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │             453 │          0 │  2.987197664237512 │            0 │
│ client_73cda7b4e4f265ea │ content_36c36abc7650d7af │            5630 │          6 │  6.724038803627149 │            3 │
│ client_73cda7b4e4f265ea │ content_a7da352b73b02668 │            4944 │         13 │  7.244843828988962 │            2 │
│ client_73cda7b4e4f265e

In [12]:
from datasets import load_dataset
content_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    streaming=False,
    split="train"
)

content_df = content_ds.to_pandas()

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

In [13]:
content_df = content_df[
    ["client_hash_id", "content_hash_id", "word_count"]
]

content_df.head()

,client_hash_id,content_hash_id,word_count
0,client_04660893ae39614a,content_004de9653278b5a4,2555.0
1,client_04660893ae39614a,content_00dc5efae381b2ab,2430.0
2,client_04660893ae39614a,content_01410f2556c327ac,2645.0
3,client_04660893ae39614a,content_019f27f634053ca7,2522.0
4,client_04660893ae39614a,content_01efa71faea45dcc,2552.0


In [14]:
feature_df = features.df().merge(
    content_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,word_count
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,0.0,NaN
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,NaN
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,NaN
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,2475.0


### Feature availability

- `gsc_impressions`: Knowable at the decision moment because it represents GSC impressions observed during March 2026.
- `gsc_clicks`: Knowable at the decision moment because it represents GSC clicks observed during March 2026.
- `gsc_avg_position`: Knowable at the decision moment because it represents GSC ranking performance observed during March 2026.
- `ga4_sessions`: Knowable at the decision moment when GA4 data is available for the content during March 2026.
- `word_count`: Knowable at the decision moment because it is content metadata associated with the content item.

### Deliberate leakage experiment

The decision moment is the end of March 2026. I define a proxy outcome using the following month: a content item is labeled as declining when its April 2026 impressions are lower than its March 2026 impressions.

For the leakage experiment, I intentionally add this label-derived column as a model feature. This represents information that would not be available at the decision moment.

In [15]:
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
march = feature_df.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

march["is_declining_label"] = (
    march["april_impressions"] < march["gsc_impressions"]
)

In [17]:
march[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "april_impressions",
        "is_declining_label"
    ]
].head()

,client_hash_id,content_hash_id,gsc_impressions,april_impressions,is_declining_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,1151.0,False
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,73.0,False
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,98.0,True
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,2275.0,False
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,6266.0,False


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = march[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "word_count",
    "is_declining_label"   # intentional leakage
]].copy()

y = march["is_declining_label"]

X = X.fillna(0)
y = y.fillna(False).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

leak_score = accuracy_score(y_test, pred)

print("Leaky score:", leak_score)

Leaky score: 1.0


In [19]:
X_honest = march[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "word_count"
]].copy()

X_honest = X_honest.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(y_test, honest_pred)

print("Honest score:", honest_score)

Honest score: 0.8031167028723147


The leaky score was artificially high because `is_declining_label` was included as a feature even though it is the target itself and depends on future April data.

After removing the leaked column, the honest score decreased. The leaked column must not be used as a model feature.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data can identify pages with declining or weak performance, but it cannot tell us whether refreshing a page will actually improve its future performance.

The dataset also has uneven data availability across clients and time. Some rows have GSC data available while GA4 data may be unavailable, so historical coverage is not balanced across all sources.

In addition, the 90-day query data uses overlapping windows, so it should not be treated as independent observations.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.